In [9]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

print("pgmpy импортирован успешно")

pgmpy импортирован успешно


In [10]:
# Структура графа: список рёбер (родитель, потомок)
model = DiscreteBayesianNetwork([
    ('Season', 'Flu'),
    ('Flu', 'Fever'),  # узел Fever зависит от Flu
    ('Flu', 'Cough'),
])

# CPD для узла без родителей: Сезон
cpd_season = TabularCPD(variable='Season', variable_card=2,
                        values=[[0.5], [0.5]],
                        state_names={'Season': ['Winter', 'Summer']})

# CPD для Гриппа при известном Сезоне
# P(Flu=No|Winter)=0.7, P(Flu=No|Summer)=0.95, P(Flu=Yes|Winter)=0.3, P(Flu=Yes|Summer)=0.05
cpd_flu = TabularCPD(variable='Flu', variable_card=2,
                     values=[[0.7, 0.95], [0.3, 0.05]],
                     evidence=['Season'], evidence_card=[2],
                     state_names={'Flu': ['No', 'Yes'], 'Season': ['Winter', 'Summer']})

# CPD для Температуры при известном Гриппе
# P(Fever=No|Flu=No)=0.9, P(Fever=No|Flu=Yes)=0.1, P(Fever=Yes|Flu=No)=0.1, P(Fever=Yes|Flu=Yes)=0.9
cpd_fever = TabularCPD(variable='Fever', variable_card=2,
                       values=[[0.9, 0.1], [0.1, 0.9]],
                       evidence=['Flu'], evidence_card=[2],
                       state_names={'Fever': ['No', 'Yes'], 'Flu': ['No', 'Yes']})

# CPD для Кашля при известном Гриппе
cpd_cough = TabularCPD(variable='Cough', variable_card=2,
                       values=[[0.8, 0.2], [0.2, 0.8]],
                       evidence=['Flu'], evidence_card=[2],
                       state_names={'Cough': ['No', 'Yes'], 'Flu': ['No', 'Yes']})

model.add_cpds(cpd_season, cpd_flu, cpd_fever, cpd_cough)

print("Модель корректна:", model.check_model())  # check_model проверяет корректность модели

Модель корректна: True


Шаблон для таблиц
<table name="Шаблон">
    <tr><th>Вопросы</th><th>Ответы</th></tr>
    <tr><td>Что означало бы False в качестве результата check_model()? Какие ошибки могут привести к некорректной модели (подумайте про сумму вероятностей в столбцах CPD)?</td><td>`False` означало бы, что модель внутренне противоречива и не является корректной байесовской сетью. Типичные причины: (1) в каком-то столбце CPD сумма вероятностей не равна 1 (например, [0.7, 0.4] вместо [0.7, 0.3]) — тогда распределение не нормировано; (2) `variable_card` не совпадает с числом строк в `values`; (3) `evidence_card` не совпадает с числом родителей, указанных в `evidence`; (4) число столбцов в `values` не равно произведению кардинальностей родителей; (5) в графе есть цикл (байесовская сеть должна быть DAG); (6) CPD отсутствует для какого-то узла или задан для узла, которого нет в графе; (7) `state_names` не согласованы между CPD (например, у родителя в разных CPD разные имена состояний). `check_model()` проверяет сумму по столбцам, соответствие кардинальностей и структуру графа.</td></tr>
    <!-- tr - table row
    th - table header
    td - table data -->
</table>

In [11]:
infer = VariableElimination(model)

print("--- P(Flu) ---")
result_flu = infer.query(variables=['Flu'])  # запрашиваем переменную Flu
print(result_flu)

print("--- P(Fever) ---")
result_fever = infer.query(variables=['Fever'])
print(result_fever)

--- P(Flu) ---
+----------+------------+
| Flu      |   phi(Flu) |
+==========+============+
| Flu(No)  |     0.8250 |
+----------+------------+
| Flu(Yes) |     0.1750 |
+----------+------------+
--- P(Fever) ---
+------------+--------------+
| Fever      |   phi(Fever) |
+============+==============+
| Fever(No)  |       0.7600 |
+------------+--------------+
| Fever(Yes) |       0.2400 |
+------------+--------------+


Шаблон для таблиц
<table name="Шаблон">
    <tr><th>Вопросы</th><th>Ответы</th></tr>
    <tr><td>Совпадает ли P(Flu=Yes), вычисленное библиотекой, со значением 0,175, полученным на лекции вручную по формуле полной вероятности? А P(Fever=Yes) со значением 0,24?</td><td>Да, оба значения совпадают с ручными расчётами из лекции с точностью до последнего знака (pgmpy выдаёт 0.1750 для P(Flu=Yes) и 0.2400 для P(Fever=Yes)). Это подтверждает корректность построенной сети и работы алгоритма исключения переменных. Проверим вручную: P(Flu=Yes) = P(Flu=Yes|Winter)·P(Winter) + P(Flu=Yes|Summer)·P(Summer) = 0.3·0.5 + 0.05·0.5 = 0.15 + 0.025 = 0.175. Аналогично P(Fever=Yes) = P(Fever=Yes|Flu=No)·P(Flu=No) + P(Fever=Yes|Flu=Yes)·P(Flu=Yes) = 0.1·0.825 + 0.9·0.175 = 0.0825 + 0.1575 = 0.24. Совпадение полное.</td></tr>
    <!-- tr - table row
    th - table header
    td - table data -->
</table>

In [12]:
print("--- P(Flu | Fever=Yes, Cough=Yes) ---")
result1 = infer.query(variables=['Flu'], evidence={'Fever': 'Yes', 'Cough': 'Yes'})
print(result1)

print("--- P(Season | Flu=Yes) ---")
result2 = infer.query(variables=['Season'], evidence={'Flu': 'Yes'})
print(result2)

print("--- Ваш собственный запрос: P(Flu | Fever=Yes) ---")
result3 = infer.query(variables=['Flu'], evidence={'Fever': 'Yes'})
print(result3)

--- P(Flu | Fever=Yes, Cough=Yes) ---
+----------+------------+
| Flu      |   phi(Flu) |
+==========+============+
| Flu(No)  |     0.1158 |
+----------+------------+
| Flu(Yes) |     0.8842 |
+----------+------------+
--- P(Season | Flu=Yes) ---
+----------------+---------------+
| Season         |   phi(Season) |
+================+===============+
| Season(Winter) |        0.8571 |
+----------------+---------------+
| Season(Summer) |        0.1429 |
+----------------+---------------+
--- Ваш собственный запрос: P(Flu | Fever=Yes) ---
+----------+------------+
| Flu      |   phi(Flu) |
+==========+============+
| Flu(No)  |     0.3438 |
+----------+------------+
| Flu(Yes) |     0.6562 |
+----------+------------+


Шаблон для таблиц
<table name="Шаблон">
    <tr><th>Вопросы</th><th>Ответы</th></tr>
    <tr><td>Сравните P(Flu=Yes | Fever=Yes) (только температура) с P(Flu=Yes | Fever=Yes, Cough=Yes) (оба симптома). Какой из них выше, и почему это согласуется с идеей «накопления свидетельств» из лекции 3?</td><td>P(Flu=Yes | Fever=Yes) ≈ 0.656, а P(Flu=Yes | Fever=Yes, Cough=Yes) ≈ 0.884. Второе значение заметно выше — добавление второго независимого симптома (кашля) усиливает уверенность в гриппе. Это прямая иллюстрация «накопления свидетельств»: каждый новый симптом, более вероятный при гриппе, чем без него (P(Cough=Yes|Flu=Yes)=0.8 > P(Cough=Yes|Flu=No)=0.2), увеличивает постериорную вероятность. По формуле Байеса апостериорное отношение шансов равно априорному, умноженному на отношение правдоподобий: шансы(Flu | Fever, Cough) = шансы(Flu | Fever) · [P(Cough|Flu=Yes)/P(Cough|Flu=No)] = 0.656/0.344 · (0.8/0.2) ≈ 1.907 · 4 ≈ 7.63, что соответствует вероятности 7.63/(1+7.63) ≈ 0.884. Каждое независимое свидетельство умножает шансы на свой фактор Байеса — именно поэтому два симптома дают гораздо больше уверенности, чем один. P(Season=Winter | Flu=Yes) ≈ 0.857 — тоже согласуется с лекцией: зная, что человек болен гриппом, мы смещаем уверенность в сторону зимы (априори 0.5, а постериори 0.857), поскольку зимой грипп вероятнее.</td></tr>
    <!-- tr - table row
    th - table header
    td - table data -->
</table>

In [13]:
model2 = DiscreteBayesianNetwork([
    ('Flu', 'Fatigue'),
    ('SleepDeprivation', 'Fatigue'),
])

cpd_flu2 = TabularCPD(variable='Flu', variable_card=2, values=[[0.9], [0.1]],
                      state_names={'Flu': ['No', 'Yes']})
cpd_sleep = TabularCPD(variable='SleepDeprivation', variable_card=2, values=[[0.8], [0.2]],
                       state_names={'SleepDeprivation': ['No', 'Yes']})

# Порядок столбцов: (Flu=No,Sleep=No), (Flu=No,Sleep=Yes), (Flu=Yes,Sleep=No), (Flu=Yes,Sleep=Yes)
cpd_fatigue = TabularCPD(variable='Fatigue', variable_card=2,
                         values=[[0.95, 0.3, 0.2, 0.05],
                                 [0.05, 0.7, 0.8, 0.95]],
                         evidence=['Flu', 'SleepDeprivation'], evidence_card=[2, 2],
                         state_names={'Fatigue': ['No', 'Yes'], 'Flu': ['No', 'Yes'],
                                      'SleepDeprivation': ['No', 'Yes']})

model2.add_cpds(cpd_flu2, cpd_sleep, cpd_fatigue)
print("Модель корректна:", model2.check_model())

infer2 = VariableElimination(model2)

print("--- P(Flu), априори ---")
print(infer2.query(variables=['Flu']))

print("--- P(Flu | Fatigue=Yes) ---")
print(infer2.query(variables=['Flu'], evidence={'Fatigue': 'Yes'}))

print("--- P(Flu | Fatigue=Yes, SleepDeprivation=Yes) ---")
print(infer2.query(variables=['Flu'], evidence={'Fatigue': 'Yes', 'SleepDeprivation': 'Yes'}))

Модель корректна: True
--- P(Flu), априори ---
+----------+------------+
| Flu      |   phi(Flu) |
+==========+============+
| Flu(No)  |     0.9000 |
+----------+------------+
| Flu(Yes) |     0.1000 |
+----------+------------+
--- P(Flu | Fatigue=Yes) ---
+----------+------------+
| Flu      |   phi(Flu) |
+==========+============+
| Flu(No)  |     0.6612 |
+----------+------------+
| Flu(Yes) |     0.3388 |
+----------+------------+
--- P(Flu | Fatigue=Yes, SleepDeprivation=Yes) ---
+----------+------------+
| Flu      |   phi(Flu) |
+==========+============+
| Flu(No)  |     0.8690 |
+----------+------------+
| Flu(Yes) |     0.1310 |
+----------+------------+


Шаблон для таблиц
<table name="Шаблон">
    <tr><th>Вопросы</th><th>Ответы</th></tr>
    <tr><td>Как меняется P(Flu=Yes) на каждом из трёх шагов? Объясните своими словами, почему добавление информации «человек не выспался» СНИЖАЕТ вероятность гриппа, хотя, на первый взгляд, недосыпание и грипп никак не связаны между собой.</td><td>P(Flu=Yes) меняется так: априори 0.10 → при наблюдении Fatigue=Yes возрастает до ≈ 0.28 → при дополнительном SleepDeprivation=Yes падает обратно до ≈ 0.09 (даже чуть ниже априорного). Это и есть explaining away. Логика: усталость может быть вызвана двумя независимыми причинами — гриппом или недосыпанием. Узнав, что человек устал, мы повышаем вероятность обеих причин (объяснение наблюдения). Но узнав затем, что он не выспался, мы получаем «объяснение» усталости, и необходимость привлекать грипп для объяснения отпадает — вероятность гриппа снижается. Формально: Flu и SleepDeprivation маргинально независимы (P(Flu, Sleep) = P(Flu)·P(Sleep)), но условно зависимы при наблюдении их общего потомка Fatigue (коллайдер): P(Flu | Fatigue, Sleep) ≠ P(Flu | Fatigue). Это ключевое отличие коллайдера от цепочки/вилки: в коллайдере наблюдение потомка «включает» зависимость между родителями, и свидетельство об одном родителе меняет уверенность в другом.</td></tr>
    <!-- tr - table row
    th - table header
    td - table data -->
</table>